In [1]:
"""
Build hierarchical parent index and parent-child relationships from chunked legal documents.

Input:  JSON file containing a list of chunks (each chunk includes document metadata and chunk fields).
Output: Three JSON files:
          - document_index.json   : parent documents, keyed by doc_id
          - chunk_index.json      : all chunks, keyed by chunk_id, each with doc_id reference
          - hierarchy.json        : nested tree: doc_id -> articles -> clauses -> points -> chunk_id list
"""

'\nBuild hierarchical parent index and parent-child relationships from chunked legal documents.\n\nInput:  JSON file containing a list of chunks (each chunk includes document metadata and chunk fields).\nOutput: Three JSON files:\n          - document_index.json   : parent documents, keyed by doc_id\n          - chunk_index.json      : all chunks, keyed by chunk_id, each with doc_id reference\n          - hierarchy.json        : nested tree: doc_id -> articles -> clauses -> points -> chunk_id list\n'

In [2]:
import json
import hashlib
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Any

In [3]:
# ----------------------------------------------------------------------
# ID generation (stable, content‑based)
# ----------------------------------------------------------------------

def get_doc_id(doc_metadata: Dict[str, Any]) -> str:
    """Generate a stable 16‑character document ID from URL or title+number."""
    url = doc_metadata.get("url", "")
    if url:
        unique = url
    else:
        unique = f"{doc_metadata.get('Tên văn bản', '')}_{doc_metadata.get('Số hiệu', '')}"
    return hashlib.md5(unique.encode("utf-8")).hexdigest()[:16]

def get_chunk_id(doc_id: str, chunk: Dict[str, Any]) -> str:
    """Generate a stable chunk ID from document ID and hierarchical position."""
    level = chunk.get("level", "unknown")
    article = chunk.get("article_number", "")
    clause = chunk.get("clause_number", "")
    point = chunk.get("point", "")
    # Normalise empty strings to "0"
    article = article if article else "0"
    clause = clause if clause else "0"
    point = point if point else "0"
    return f"{doc_id}__{level}__art{article}__cl{clause}__pt{point}"

In [4]:
# ----------------------------------------------------------------------
# Build indexes and hierarchy
# ----------------------------------------------------------------------

def build_indexes(chunks: List[Dict[str, Any]]) -> tuple:
    doc_index = {}
    chunk_index = {}
    hierarchy = {}

    for chunk in chunks:
        # Extract document metadata (non‑chunk fields)
        doc_metadata = {k: v for k, v in chunk.items()
                        if k not in ["chunk_content", "level", "article_number",
                                     "article_title", "clause_number", "clause_title",
                                     "point", "start_char", "end_char", "subpart"]}
        doc_id = get_doc_id(doc_metadata)

        # Document index (upsert)
        doc_index[doc_id] = {
            "doc_id": doc_id,
            "url": doc_metadata.get("url"),
            "title": doc_metadata.get("Tên văn bản"),
            "number": doc_metadata.get("Số hiệu"),
            "last_updated": datetime.now().isoformat(),
            **doc_metadata
        }

        # Chunk index
        chunk_entry = chunk.copy()
        chunk_entry["doc_id"] = doc_id
        chunk_id = get_chunk_id(doc_id, chunk)
        chunk_entry["chunk_id"] = chunk_id
        chunk_index[chunk_id] = chunk_entry

        # Hierarchy building – use plain dicts and convert keys to strings
        art = str(chunk.get("article_number") or "0")
        clause = str(chunk.get("clause_number") or "0")
        point = str(chunk.get("point") or "0")

        # Ensure nested dicts exist
        if doc_id not in hierarchy:
            hierarchy[doc_id] = {}
        if art not in hierarchy[doc_id]:
            hierarchy[doc_id][art] = {}
        if clause not in hierarchy[doc_id][art]:
            hierarchy[doc_id][art][clause] = {}
        if point not in hierarchy[doc_id][art][clause]:
            hierarchy[doc_id][art][clause][point] = []

        hierarchy[doc_id][art][clause][point].append(chunk_id)

    return doc_index, chunk_index, hierarchy

def save_indexes(doc_index, chunk_index, hierarchy, output_dir="index"):
    """Save the three indexes as JSON files."""
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    with open(out_path / "document_index.json", "w", encoding="utf-8") as f:
        json.dump(doc_index, f, ensure_ascii=False, indent=2)
    with open(out_path / "chunk_index.json", "w", encoding="utf-8") as f:
        json.dump(chunk_index, f, ensure_ascii=False, indent=2)
    # Convert defaultdict to plain dict for JSON
    hierarchy_plain = {
        doc_id: {art: {cl: {pt: chunk_ids for pt, chunk_ids in cl_dict.items()}
                      for cl, cl_dict in art_dict.items()}
                for art, art_dict in doc_dict.items()}
        for doc_id, doc_dict in hierarchy.items()
    }
    with open(out_path / "hierarchy.json", "w", encoding="utf-8") as f:
        json.dump(hierarchy_plain, f, ensure_ascii=False, indent=2)

    print(f"Saved document index: {len(doc_index)} documents")
    print(f"Saved chunk index: {len(chunk_index)} chunks")
    print(f"Saved hierarchy tree")

In [5]:
# ----------------------------------------------------------------------
# Optional: delete documents not in the current run (full refresh)
# ----------------------------------------------------------------------

def clean_stale_documents(current_doc_ids, previous_doc_index_path, previous_chunk_index_path, output_dir):
    """Remove documents that no longer exist in the current run from the indexes."""
    if not Path(previous_doc_index_path).exists():
        return
    with open(previous_doc_index_path, "r", encoding="utf-8") as f:
        old_doc_index = json.load(f)
    with open(previous_chunk_index_path, "r", encoding="utf-8") as f:
        old_chunk_index = json.load(f)

    new_doc_index = {k:v for k,v in old_doc_index.items() if k in current_doc_ids}
    new_chunk_index = {k:v for k,v in old_chunk_index.items() if v.get("doc_id") in current_doc_ids}

    # Overwrite with cleaned versions
    with open(output_dir / "document_index.json", "w", encoding="utf-8") as f:
        json.dump(new_doc_index, f, ensure_ascii=False, indent=2)
    with open(output_dir / "chunk_index.json", "w", encoding="utf-8") as f:
        json.dump(new_chunk_index, f, ensure_ascii=False, indent=2)

    print(f"Removed {len(old_doc_index)-len(new_doc_index)} stale documents")

In [ ]:
# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------

def main():
    import argparse
    parser = argparse.ArgumentParser(description="Build hierarchical parent index from chunked JSON.")
    parser.add_argument("--input", type=str, required=True,
                        help="Path to chunked JSON file (list of chunks).")
    parser.add_argument("--output-dir", type=str, default="index",
                        help="Directory to save index files (default: index/).")
    parser.add_argument("--clean-stale", action="store_true",
                        help="Remove documents not present in the current input (full refresh).")
    args = parser.parse_args()

    # Load input chunks
    with open(args.input, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"Loaded {len(chunks)} chunks from {args.input}")

    # Build indexes
    doc_index, chunk_index, hierarchy = build_indexes(chunks)

    # Save new indexes (temporarily, if --clean-stale will overwrite)
    save_indexes(doc_index, chunk_index, hierarchy, args.output_dir)

    # Optionally clean stale documents
    if args.clean_stale:
        current_doc_ids = set(doc_index.keys())
        clean_stale_documents(
            current_doc_ids,
            Path(args.output_dir) / "document_index.json",
            Path(args.output_dir) / "chunk_index.json",
            Path(args.output_dir)
        )

In [9]:
from tqdm import tqdm
from pathlib import Path
import json

def main_notebook(input_file="all_chunks.json", output_dir="index", clean_stale=False):
    """
    Build hierarchical parent index from a chunked JSON file.
    
    Parameters:
        input_file (str): Path to the chunked JSON file (list of chunks).
        output_dir (str): Directory to save index files.
        clean_stale (bool): If True, remove documents not in current input.
    """
    # Load input chunks
    with open(input_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"Loaded {len(chunks)} chunks from {input_file}")

    # Build indexes with progress bar
    doc_index = {}
    chunk_index = {}
    hierarchy = {}

    for chunk in tqdm(chunks, desc="Indexing chunks"):
        # Extract document metadata
        doc_metadata = {k: v for k, v in chunk.items()
                        if k not in ["chunk_content", "level", "article_number",
                                     "article_title", "clause_number", "clause_title",
                                     "point", "start_char", "end_char", "subpart"]}
        doc_id = get_doc_id(doc_metadata)

        # Document index (upsert)
        from datetime import datetime
        doc_index[doc_id] = {
            "doc_id": doc_id,
            "url": doc_metadata.get("url"),
            "title": doc_metadata.get("Tên văn bản"),
            "number": doc_metadata.get("Số hiệu"),
            "last_updated": datetime.now().isoformat(),
            **doc_metadata
        }

        # Chunk index
        chunk_entry = chunk.copy()
        chunk_entry["doc_id"] = doc_id
        chunk_id = get_chunk_id(doc_id, chunk)
        chunk_entry["chunk_id"] = chunk_id
        chunk_index[chunk_id] = chunk_entry

        # Hierarchy
        art = str(chunk.get("article_number") or "0")
        clause = str(chunk.get("clause_number") or "0")
        point = str(chunk.get("point") or "0")

        if doc_id not in hierarchy:
            hierarchy[doc_id] = {}
        if art not in hierarchy[doc_id]:
            hierarchy[doc_id][art] = {}
        if clause not in hierarchy[doc_id][art]:
            hierarchy[doc_id][art][clause] = {}
        if point not in hierarchy[doc_id][art][clause]:
            hierarchy[doc_id][art][clause][point] = []
        hierarchy[doc_id][art][clause][point].append(chunk_id)

    # Save indexes
    save_indexes(doc_index, chunk_index, hierarchy, output_dir)

    # Optionally clean stale documents
    if clean_stale:
        current_doc_ids = set(doc_index.keys())
        clean_stale_documents(
            current_doc_ids,
            Path(output_dir) / "document_index.json",
            Path(output_dir) / "chunk_index.json",
            Path(output_dir)
        )

In [ ]:
# ==================================================
# Execute: change the input file path if needed
# ==================================================
main_notebook(input_file="../../data/legal_document/all_chunks.json", output_dir="../../data/index/", clean_stale=False)

Loaded 3315 chunks from ../../data/legal_document/all_chunks.json


Indexing chunks: 100%|██████████| 3315/3315 [00:00<00:00, 91985.83it/s]


Saved document index: 12 documents
Saved chunk index: 3260 chunks
Saved hierarchy tree


## 2. HOW TO USE

In [15]:
with open("index/document_index.json", "r", encoding="utf-8") as f:
    doc_index = json.load(f)
with open("index/chunk_index.json", "r", encoding="utf-8") as f:
    chunk_index = json.load(f)
with open("index/hierarchy.json", "r", encoding="utf-8") as f:
    hierarchy = json.load(f)

print(f"Loaded {len(doc_index)} documents, {len(chunk_index)} chunks")

Loaded 12 documents, 3260 chunks


In [16]:
if chunk_index:
    sample_chunk_id = list(chunk_index.keys())[0]
    print(f"Sample chunk ID: {sample_chunk_id}")
else:
    raise ValueError("No chunks found in index")

Sample chunk ID: ac2b53d52a79dc25__clause__art1__cl1__pt0


### 2.1. Find parent chunks

In [21]:
# Cell 3: Tìm văn bản cha của chunk mẫu
chunk = chunk_index.get(sample_chunk_id)
if chunk:
    doc_id = chunk["doc_id"]
    parent = doc_index.get(doc_id)
    print("=" * 60)
    print("TÌM VĂN BẢN CHA CỦA CHUNK")
    print(f"Chunk ID: {sample_chunk_id}")
    print(f"Doc ID: {doc_id}")
    if parent:
        print(f"Tên văn bản: {parent.get('title', 'N/A')}")
        print(f"Số hiệu: {parent.get('number', 'N/A')}")
        print(f"URL: {parent.get('url', 'N/A')}")
    else:
        print("Không tìm thấy văn bản cha!")
    print(f"Nội dung chunk (300 ký tự đầu):\n{chunk.get('chunk_content', '')}")
    print("=" * 60)

TÌM VĂN BẢN CHA CỦA CHUNK
Chunk ID: ac2b53d52a79dc25__clause__art1__cl1__pt0
Doc ID: ac2b53d52a79dc25
Tên văn bản: Luật Sửa đổi, bổ sung một số điều của Luật Doanh nghiệp số 76/2025/QH15
Số hiệu: 76/2025/QH15
URL: https://vbpl.vn/van-ban/chi-tiet/luat-sua-doi-bo-sung-mot-so-dieu-cua-luat-doanh-nghiep-so-76-2025-qh15--179095
Nội dung chunk (300 ký tự đầu):
1. Sửa đổi, bổ sung một số khoản của Điều 4 như sau:
a) Sửa đổi, bổ sung khoản 5 như sau:
"5. Cổ tức là khoản lợi nhuận sau thuế được trả cho mỗi cổ phần bằng tiền hoặc bằng tài sản khác.";
b) Sửa đổi, bổ sung khoản 14 như sau:
"14. Giá thị trường của phần vốn góp hoặc cổ phần là:
a) Giá giao dịch bình quân trong vòng 30 ngày liền kề trước ngày xác định giá hoặc giá thỏa thuận giữa người bán và người mua hoặc giá do một tổ chức thẩm định giá xác định đối với cổ phiếu niêm yết, đăng ký giao dịch trên hệ thống giao dịch chứng khoán;
b) Giá giao dịch trên thị trường tại thời điểm liền kề trước đó hoặc giá thỏa thuận giữa người bán và ngư

### 2.2. Find all chunks of a document

In [23]:
# Cell 4: Lấy tất cả chunk của văn bản đó
if doc_id:
    chunks_of_doc = [ch for ch in chunk_index.values() if ch.get("doc_id") == doc_id]
    print(f"Số chunk của văn bản '{parent.get('title', '')}': {len(chunks_of_doc)}")
    print("10 chunk đầu tiên:")
    for i, ch in enumerate(chunks_of_doc[:10]):
        print(f"  {i+1}. Chunk ID: {ch['chunk_id']} | Level: {ch.get('level')} | Article: {ch.get('article_number')}")
    print("=" * 60)

Số chunk của văn bản 'Luật Sửa đổi, bổ sung một số điều của Luật Doanh nghiệp số 76/2025/QH15': 31
10 chunk đầu tiên:
  1. Chunk ID: ac2b53d52a79dc25__clause__art1__cl1__pt0 | Level: clause | Article: 1
  2. Chunk ID: ac2b53d52a79dc25__clause__art1__cl2__pt0 | Level: clause | Article: 1
  3. Chunk ID: ac2b53d52a79dc25__clause__art1__cl3__pt0 | Level: clause | Article: 1
  4. Chunk ID: ac2b53d52a79dc25__clause__art1__cl4__pt0 | Level: clause | Article: 1
  5. Chunk ID: ac2b53d52a79dc25__clause__art1__cl5__pt0 | Level: clause | Article: 1
  6. Chunk ID: ac2b53d52a79dc25__clause__art1__cl6__pt0 | Level: clause | Article: 1
  7. Chunk ID: ac2b53d52a79dc25__clause__art1__cl7__pt0 | Level: clause | Article: 1
  8. Chunk ID: ac2b53d52a79dc25__clause__art1__cl8__pt0 | Level: clause | Article: 1
  9. Chunk ID: ac2b53d52a79dc25__clause__art1__cl9__pt0 | Level: clause | Article: 1
  10. Chunk ID: ac2b53d52a79dc25__clause__art1__cl10__pt0 | Level: clause | Article: 1


### 2.3. Retrieve all chunks

In [25]:
# Cell 5: Duyệt cấu trúc cây của văn bản đó
if doc_id:
    doc_hierarchy = hierarchy.get(doc_id, {})
    if not doc_hierarchy:
        print("Không có cấu trúc phân cấp cho văn bản này.")
    else:
        print(f"Cấu trúc cây của văn bản '{parent.get('title', '')}':")
        for article, clauses in doc_hierarchy.items():
            print(f"\nĐiều {article}")
            for clause, points in clauses.items():
                print(f"  Khoản {clause}")
                # for point, chunk_ids in points.items():
                #     print(f"    Điểm {point}: {len(chunk_ids)} chunk(s) - IDs: {chunk_ids[:2]}{'...' if len(chunk_ids)>2 else ''}")
    print("=" * 60)

Cấu trúc cây của văn bản 'Luật Sửa đổi, bổ sung một số điều của Luật Doanh nghiệp số 76/2025/QH15':

Điều 1
  Khoản 1
  Khoản 2
  Khoản 3
  Khoản 4
  Khoản 5
  Khoản 6
  Khoản 7
  Khoản 8
  Khoản 9
  Khoản 10
  Khoản 11
  Khoản 12
  Khoản 13
  Khoản 14
  Khoản 15
  Khoản 16
  Khoản 17
  Khoản 18
  Khoản 19
  Khoản 20
  Khoản 21
  Khoản 22
  Khoản 23
  Khoản 24
  Khoản 25
  Khoản 26
  Khoản 27
  Khoản 28

Điều 2
  Khoản 0

Điều 3
  Khoản 1
  Khoản 2
